Load the Data: Use PyTorch's ImageFolder to load the data. Since the data is structured in a way where each folder name corresponds to a category, this works well with ImageFolder

In [1]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torchvision import models

# Define transforms for training and validation
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load the dataset
train_dir = 'C:/Users/shres/Documents/dataset/RoadSaW-150_l/train'
val_dir = 'C:/Users/shres/Documents/dataset/RoadSaW-150_l/validation'
test_dir = 'C:/Users/shres/Documents/dataset/RoadSaW-150_l/test'

train_dataset = datasets.ImageFolder(train_dir, transform=transform)
val_dataset = datasets.ImageFolder(val_dir, transform=transform)
test_dataset = datasets.ImageFolder(test_dir, transform=transform)

# Data loaders for batching
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


Load Pre-trained ResNet Model: Use a pre-trained ResNet model for transfer learning. Fine-tune the model to classify your 15 categories

In [2]:
# Load pre-trained ResNet model
model = models.resnet50(pretrained=True)

# Modify the final layer to match the number of categories (15 in this case)
model.fc = torch.nn.Linear(model.fc.in_features, 15)

# Move model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)


c:\Users\shres\anaconda3\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\shres\anaconda3\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to C:\Users\shres/.cache\torch\hub\checkpoints\resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:15<00:00, 6.45MB/s]


Set Up the Loss Function and Optimizer: Use a loss function like CrossEntropyLoss for multi-class classification and an optimizer such as Adam.

In [3]:
import torch.optim as optim

criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


Train the Model: Train the model using the training data, and validate it on the validation set after each epoch.

In [4]:
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        # Zero the gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # Track accuracy
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total
    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.2f}%')

    # Validation after each epoch
    model.eval()
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_acc = 100 * val_correct / val_total
    print(f'Validation Accuracy: {val_acc:.2f}%')


Epoch 1/10, Loss: 0.8730, Accuracy: 65.85%
Validation Accuracy: 51.57%
Epoch 2/10, Loss: 0.5903, Accuracy: 75.55%
Validation Accuracy: 66.76%
Epoch 3/10, Loss: 0.5011, Accuracy: 79.07%
Validation Accuracy: 58.48%
Epoch 4/10, Loss: 0.4670, Accuracy: 80.50%
Validation Accuracy: 76.82%
Epoch 5/10, Loss: 0.4122, Accuracy: 82.62%
Validation Accuracy: 69.68%
Epoch 6/10, Loss: 0.3843, Accuracy: 83.79%
Validation Accuracy: 73.44%
Epoch 7/10, Loss: 0.3614, Accuracy: 84.70%
Validation Accuracy: 64.79%
Epoch 8/10, Loss: 0.3361, Accuracy: 86.22%
Validation Accuracy: 72.62%
Epoch 9/10, Loss: 0.3143, Accuracy: 86.90%
Validation Accuracy: 75.51%
Epoch 10/10, Loss: 0.2815, Accuracy: 88.44%
Validation Accuracy: 68.08%


Test the Model: After training, test the model on the test dataset to evaluate its performance.

In [5]:
model.eval()
test_correct = 0
test_total = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()

test_acc = 100 * test_correct / test_total
print(f'Test Accuracy: {test_acc:.2f}%')


Test Accuracy: 65.16%
